# ICS40125 - Laboratorio N°08

Machine Learning no supervisado: KMeans y reducción de dimensionalidad.

## Clustering — Vehículos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

%matplotlib inline
sns.set_palette("deep", desat=.6)

df = pd.read_csv("https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/vehiculos_procesado_con_grupos.csv", sep=",")\
       .drop(["fabricante","modelo","transmision","traccion","clase","combustible","consumo"], axis=1)
df.head()

### 1. Normalizar datos

In [ ]:
# separar numericas y categoricas
numericas = df.select_dtypes(include=[np.number])
categoricas = df.select_dtypes(exclude=[np.number])

# rellenar nulos numericos con el promedio de la columna
numericas = numericas.fillna(numericas.mean())

# normalizar con MinMaxScaler
scaler = MinMaxScaler()
num_norm = pd.DataFrame(scaler.fit_transform(numericas), columns=numericas.columns, index=df.index)

# codificar categoricas con get_dummies (one-hot: cada categoria pasa a una columna 0/1)
cat_dummies = pd.get_dummies(categoricas)

# juntar ambos
df_procesado = pd.concat([num_norm, cat_dummies], axis=1)
df_procesado.head()

### 2. Ajuste con KMeans (8 clusters)

In [ ]:
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df_procesado)

# centroides
print("Forma centroides:", kmeans.cluster_centers_.shape)
df['cluster'].value_counts().sort_index()

In [ ]:
# resumen por cluster: promedio de numericas
resumen_num = df.groupby('cluster')[numericas.columns].mean()
resumen_num

In [ ]:
# moda de las categoricas por cluster
resumen_cat = df.groupby('cluster')[categoricas.columns].agg(lambda s: s.mode().iloc[0])
resumen_cat

### 3. Número de clusters (regla del codo)

In [ ]:
ks = [5, 10, 20, 30, 50, 75, 100, 200, 300]
inercias = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_procesado)
    inercias.append(km.inertia_)

plt.figure(figsize=(10,5))
plt.plot(ks, inercias, marker='o')
plt.xlabel('numero de clusters'); plt.ylabel('inercia')
plt.title('Regla del codo')
plt.show()
# La inercia cae rapido al inicio y luego se aplana: el "codo" esta en un valor
# relativamente bajo (alrededor de 20-30 clusters), donde agregar mas grupos
# ya casi no reduce la inercia.

## Reducción de Dimensionalidad — Wine

In [ ]:
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

dataset = load_wine()
features = dataset.feature_names
wine = pd.DataFrame(dataset.data, columns=features)
wine['wine_class'] = dataset.target

# estandarizar antes de PCA/t-SNE
X = StandardScaler().fit_transform(wine[features])
y = wine['wine_class']
wine.head()

### 1. PCA

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X)

# varianza explicada acumulada
var_acum = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8,4))
plt.plot(range(1, len(var_acum)+1), var_acum, marker='o')
plt.axhline(0.9, color='r', linestyle='--')
plt.xlabel('componentes'); plt.ylabel('varianza acumulada')
plt.title('PCA - varianza explicada')
plt.show()

n_90 = np.argmax(var_acum >= 0.9) + 1
print("Componentes para 90% de varianza:", n_90)

In [ ]:
# proyeccion en 2D
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, palette='deep')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Vinos proyectados con PCA')
plt.show()
# Las 3 variedades se separan bastante bien en las dos primeras componentes.

In [ ]:
# loadings: influencia de cada variable en PC1 y PC2
loadings = pd.DataFrame(pca.components_[:2].T, columns=['PC1','PC2'], index=features)
loadings.sort_values('PC1', key=abs, ascending=False).head()

### 2. t-SNE

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18,5))
for ax, perp in zip(axs, [5, 30, 50]):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
    X_tsne = tsne.fit_transform(X)
    sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y, palette='deep', ax=ax, legend=False)
    ax.set_title(f'perplexity={perp}')
plt.show()
# t-SNE tambien separa las 3 clases; la perplexity cambia lo compacto de los grupos.

### 3. Comparación PCA vs t-SNE

- **PCA** es lineal, rápido e interpretable (podemos ver qué variables pesan en cada componente y cuánta varianza explican). Es útil como paso previo para otros modelos.
- **t-SNE** es no lineal y preserva vecindades locales, por lo que suele mostrar clústeres más definidos, pero es más lento y no es interpretable (los ejes no tienen significado).

En Wine ambas técnicas separan bien las 3 variedades. Para explorar y visualizar clústeres conviene t-SNE; para interpretar y reducir dimensiones antes de un modelo conviene PCA. En general, reducir dimensionalidad ayuda a visualizar patrones, quitar ruido y simplificar los modelos.